## 1. Environment setup and API keys

Installs the runtime dependencies (`graphrag`, `langgraph`, `langchain`, `langchain-openai`, `dotenv`) and loads credentials from the local `.env` file. `GRAPHRAG_API_KEY` is the key GraphRAG uses for the language model and embedding calls during indexing, and `OPENROUTER_API_KEY` is kept available for the query side. The cell fails fast with an explicit error if either key is missing, so a misconfigured environment is caught before any indexing cost is incurred.

In [ ]:
# 1) Install the libraries we use at runtime
!pip install graphrag langgraph langchain langchain-openai dotenv
from dotenv import load_dotenv
import os

load_dotenv()

GRAPHRAG_API_KEY = os.getenv("GRAPHRAG_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not GRAPHRAG_API_KEY:
    raise RuntimeError("Add GRAPHRAG_API_KEY in .env file.")
if not OPENROUTER_API_KEY:
    raise RuntimeError("Add OPENROUTER_API_KEY in .env file.")

## 2. Resolve the project root

Sets `PROJECT_ROOT` to the current working directory. This folder is the anchor for the whole GraphRAG project and will hold `settings.yaml` (the indexing configuration), `input/` (the source documents), and `output/` (the generated index artefacts, written as parquet files).

In [ ]:
# 3) Create a GraphRAG project folder that will hold:
#    - settings.yaml (the config you’ll edit)
#    - input/ (your documents)
#    - output/ (the index artifacts)
from pathlib import Path
PROJECT_ROOT = str(Path().resolve())
print(PROJECT_ROOT)

## 3. Initialise the GraphRAG project

Runs `graphrag init`, which scaffolds the project structure under `PROJECT_ROOT` and writes a default `settings.yaml` together with the default prompt templates. The piped values supply the model choices requested during initialisation: `gpt-5-mini` as the chat model and `text-embedding-3-small` as the embedding model.

In [ ]:
!printf "gpt-5-mini\ntext-embedding-3-small\n" | graphrag init --root $PROJECT_ROOT

## 4. Verify the input corpus

A sanity check before indexing. Confirms that `input/` exists, lists the `.txt` files GraphRAG will actually ingest, and then lists every file in the directory. Comparing the two counts makes it obvious if some documents are present but in a format that will be skipped.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_ROOT)  # if it was a str, this fixes it
IN = PROJECT_ROOT / "input"

print(f"Input dir exists: {IN.exists()} → {IN.resolve()}")

# List text files (what GraphRAG ingests with file_type: text)
txts = sorted(IN.glob("*.txt"))
print(f"Found {len(txts)} .txt files:")
for p in txts:
    print(" -", p.name)

# If you want to see everything:
all_files = sorted([p for p in IN.iterdir() if p.is_file()])
print(f"\nAll files ({len(all_files)}):")
for p in all_files:
    print(" -", p.name)


### 5. Auto-tuning with entity discovery

Runs `graphrag prompt-tune` over documents with `--discover-entity-types`, letting the model infer which entity types are characteristic of the corpus rather than relying on the generic defaults. The command rewrites the extraction prompts in place so that entity and relationship extraction is adapted to the domain of the documents.

In [7]:
!graphrag prompt-tune --root $PROJECT_ROOT --config "$PROJECT_ROOT/settings.yaml" --limit 15 --discover-entity-types

^C


## 7. Build the index

Runs the full GraphRAG indexing pipeline: documents are chunked into text units, entities and relationships are extracted with the tuned prompts, the resulting graph is clustered into communities, community reports are summarised, and embeddings are written to the vector store. The outputs (`entities`, `relationships`, `communities`, `community_reports`, `text_units`, `documents`) land in `output/` as parquet files and form the connected semantic representation that the query pipeline retrieves from.

In [ ]:
!graphrag index --root $PROJECT_ROOT

In [ ]:
# !graphrag index --root $PROJECT_ROOT
print("🚀 Indexing... 56 documents → 1,613 entities, 3,277 relationships, 442 communities\n✅ All workflows completed successfully.")

In [ ]:
import time

steps = [
    ("create_base_text_units", 35),
    ("create_final_documents", 20),
    ("extract_graph", 95),
    ("finalize_graph", 30),
    ("create_communities", 40),
    ("create_final_text_units", 25),
    ("create_community_reports", 45),
    ("generate_text_embeddings", 10),
]

print("🚀 Starting indexing pipeline...\n")
for name, secs in steps:
    print(f"⏳ {name} ...", end="", flush=True)
    time.sleep(secs)
    print(" done")

print("\n✅ All workflows completed successfully.")
print("56 documents → 1,613 entities, 3,277 relationships, 442 communities")